# Ideal reference: XEB OpenQASM (`things_to_run_on_ibm`)

Noiseless **statevector** simulation of each `xeb_d01_c*.qasm` file (depth tag `d01` in the filename). Use the CSV to compare hardware bitstring statistics against the **ideal Born distribution** (no device noise, no readout).

**Install (once):** `pip install qiskit qiskit_qasm3_import`

**Outputs:** `xeb_ideal_reference_from_qasm.csv` in this folder.

**Note:** Global parity $\langle Z^{\otimes 10}\rangle$ is near **zero** for these random circuits; the useful columns are entropy / purity / top outcome / mean Hamming weight of the ideal distribution.


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from qiskit import qasm3
from qiskit.quantum_info import SparsePauliOp, Statevector

def _qasm_dir() -> Path:
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        cand = base / "qnn_code" / "things_to_run_on_ibm"
        if cand.is_dir():
            return cand
    raise FileNotFoundError(
        "Could not find experiments/things_to_run_on_ibm — run from repo root or analysis/"
    )


QASM_DIR = _qasm_dir()


def _out_dir() -> Path:
    here = Path.cwd().resolve()
    if here.name == "code_data_analysis":
        return here
    sub = here / "code_data_analysis"
    if sub.is_dir():
        return sub
    return here


OUT_CSV = _out_dir() / "xeb_ideal_reference_from_qasm.csv"


def _distribution_stats(probs: dict[str, float], n: int) -> dict:
    p = np.array(list(probs.values()), dtype=float)
    p = p[p > 0]
    entropy = float(-(p * np.log2(p)).sum())
    hog = max(probs.items(), key=lambda kv: kv[1])
    mean_hw = 0.0
    for bitstr, pr in probs.items():
        mean_hw += pr * bitstr.count("1")
    return {
        "ideal_entropy_bits": entropy,
        "ideal_linear_xeb_purity": float((2**n) * float(np.sum(np.array(list(probs.values())) ** 2))),
        "ideal_top_bitstring": str(hog[0]),
        "ideal_top_prob": float(hog[1]),
        "ideal_mean_hamming": float(mean_hw),
    }


def analyse_xeb_qasm(path: Path) -> dict:
    text = path.read_text()
    qc = qasm3.loads(text)
    qc_u = qc.remove_final_measurements(inplace=False)
    n = qc_u.num_qubits
    sv = Statevector(qc_u)
    obs = SparsePauliOp("Z" * n)
    ev = complex(sv.expectation_value(obs)).real
    probs = {str(k): float(v) for k, v in sv.probabilities_dict().items()}
    stats = _distribution_stats(probs, n)
    m = re.search(r"xeb_(d\d+)_(c\d+)", path.name, re.I)
    depth_key = m.group(1).lower() if m else ""
    instance_key = m.group(2).lower() if m else ""
    row = {
        "qasm_file": path.name,
        "qasm_path": str(path),
        "depth_key": depth_key,
        "instance_key": instance_key,
        "n_qubits": n,
        "ideal_Z_all_expectation": ev,
    }
    row.update(stats)
    return row


paths = sorted(QASM_DIR.glob("xeb_*.qasm"))
assert paths, f"No xeb_*.qasm under {QASM_DIR}"
rows = [analyse_xeb_qasm(p) for p in paths]
df = pd.DataFrame(rows).sort_values(["depth_key", "instance_key"]).reset_index(drop=True)
df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV.resolve()}  ({len(df)} rows)")
df


## Using this with hardware CSVs

Join `xeb_ideal_reference_from_qasm.csv` on `qasm_file`, or on `depth_key` + `instance_key` if you expand filenames the same way. Use `ideal_entropy_bits`, `ideal_linear_xeb_purity`, and `ideal_top_*` as references for distribution-level metrics; $\langle Z^{10}\rangle$ is near zero for these random circuits and is usually **not** the headline XEB statistic.